# 🛞 Formula 1 Pit Stop Analysis

This notebook investigates the impact of pit stop performance and strategy on Formula 1 race outcomes.

### Analyses
- Fastest Pit Crews by Constructor
- Pit Stop Time vs Race Performance
- Pit Stop Strategy Effectiveness

The objective is to understand how operational efficiency and race strategy influence competitive performance.

In [2]:
# ============================================================
# NOTEBOOK 5: Pit Stop Analysis
# ============================================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

master = pd.read_csv("/content/drive/MyDrive/f1-analytics/data/processed/processedmaster_df.csv")
pit_stops = pd.read_csv("/content/drive/MyDrive/f1-analytics/data/processed/processedpit_stops_clean.csv")
pit_agg = pd.read_csv("/content/drive/MyDrive/f1-analytics/data/processed/processedpit_agg.csv")

# Add constructor info to pit stops via master
pit_with_info = pd.merge(
    pit_stops,
    master[['raceId', 'driverId', 'constructor_name', 'year', 'race_name']].drop_duplicates(),
    on=['raceId', 'driverId'],
    how='left'
)

modern_pits = pit_with_info[pit_with_info['year'] >= 2010].copy()

In [3]:
# ── ANALYSIS 1: FASTEST PIT CREWS BY CONSTRUCTOR ──────────────
#
# WHY: Pit stop speed is a TEAM metric (mechanics work as a unit).
# This directly measures constructor investment and training.
# Under pressure moments — a 0.5 second stop vs 3 second stop
# can mean the difference between a win and a podium loss.

crew_speed = (
    modern_pits.groupby('constructor_name')['duration']
    .agg(['mean', 'min', 'count'])
    .reset_index()
)
crew_speed.columns = ['Constructor', 'Avg_Stop_Time', 'Fastest_Stop', 'Total_Stops']
crew_speed = crew_speed[crew_speed['Total_Stops'] >= 50]  # enough stops for reliability
crew_speed = crew_speed.sort_values('Avg_Stop_Time')

fig8 = px.bar(
    crew_speed.head(15),
    x='Constructor',
    y='Avg_Stop_Time',
    title='Average Pit Stop Time by Constructor (2010-Present) — Lower is Better',
    color='Avg_Stop_Time',
    color_continuous_scale='RdYlGn_r',
    text=crew_speed.head(15)['Avg_Stop_Time'].round(2)
)
fig8.update_layout(
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig8.show()

In [4]:
# ── ANALYSIS 2: DOES PIT STOP TIME CORRELATE WITH RACE RESULT? ─
#
# WHY: This is a HYPOTHESIS TEST — the most analytical thing you
# can do in EDA. We're asking: "Do faster pit stops lead to better
# finishing positions?" If yes, it proves pit crew investment matters.
# This shows correlation analysis skills.

# Merge pit aggregates with final race position
pit_performance = pd.merge(
    pit_agg,
    master[['raceId', 'driverId', 'positionOrder', 'year']],
    on=['raceId', 'driverId'],
    how='left'
)
pit_performance = pit_performance[pit_performance['year'] >= 2010]
pit_performance = pit_performance.dropna(subset=['avg_stop_time', 'positionOrder'])

fig9 = px.scatter(
    pit_performance.sample(3000, random_state=42),  # sample for performance
    x='avg_stop_time',
    y='positionOrder',
    opacity=0.4,
    trendline='ols',    # add a linear regression trendline
    title='Pit Stop Time vs Race Finishing Position (Sample of 3000 Races)',
    labels={
        'avg_stop_time': 'Avg Pit Stop Duration (seconds)',
        'positionOrder': 'Finishing Position'
    }
)

# NOTE: In plotly, y-axis is flipped for position (1=best)
# So we invert the y-axis: lower number = better position = at top
fig9.update_yaxes(autorange='reversed')
fig9.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig9.show()

# Compute Pearson correlation for the caption
corr = pit_performance['avg_stop_time'].corr(pit_performance['positionOrder'])
print(f"Correlation between pit stop time and finishing position: {corr:.3f}")
print("Interpretation: Positive correlation means slower pits → worse finish position")

Correlation between pit stop time and finishing position: 0.161
Interpretation: Positive correlation means slower pits → worse finish position


In [5]:
# ── ANALYSIS 3: PIT STOP STRATEGY — 1 vs 2 vs 3 STOP RACES ───
#
# WHY: Strategy choice (how many stops) is a key differentiator.
# Teams model whether to stop once or twice based on tire deg,
# track position, etc. Do more stops help or hurt?

stops_vs_position = pd.merge(
    pit_agg[['raceId', 'driverId', 'total_stops']],
    master[['raceId', 'driverId', 'positionOrder', 'year']],
    on=['raceId', 'driverId'],
    how='left'
)
stops_vs_position = stops_vs_position[stops_vs_position['year'] >= 2010]

# Average finishing position by number of stops
strategy_summary = (
    stops_vs_position[stops_vs_position['total_stops'].between(1, 4)]
    .groupby('total_stops')['positionOrder']
    .mean()
    .reset_index()
)
strategy_summary.columns = ['Total_Stops', 'Avg_Finishing_Position']
strategy_summary['Avg_Finishing_Position'] = strategy_summary['Avg_Finishing_Position'].round(2)

fig10 = px.bar(
    strategy_summary,
    x='Total_Stops',
    y='Avg_Finishing_Position',
    title='Average Finishing Position by Pit Stop Strategy (Lower = Better)',
    text='Avg_Finishing_Position',
    color='Avg_Finishing_Position',
    color_continuous_scale='RdYlGn_r'
)
fig10.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig10.show()

# Key Findings

***Red Bull and Mercedes*** demonstrate the highest pit stop efficiency among constructors. Faster pit stops show a measurable association with stronger race results, while teams with slower pit stops tend to finish lower on average. Strategy analysis suggests that two-stop races provide the most effective balance between tire management and time loss, whereas excessive pit stops often lead to poorer finishing positions.

1. Red Bull and Mercedes Have the Fastest Pit Crews

The constructor analysis shows that Red Bull (23.7s) and Mercedes (23.71s) recorded the lowest average pit stop times among all teams analyzed.

Insight:

Elite teams not only build fast cars but also maintain highly efficient pit crews, gaining valuable seconds during races.

2. Smaller Teams Lag Behind in Pit Stop Efficiency

Teams such as Caterham (24.81s), Alpine F1 Team (24.75s), and RB F1 Team (24.73s) recorded the slowest average pit stop times.

Insight:

Operational efficiency appears to be a competitive advantage, with top teams generally executing faster pit stops than lower-performing teams.

3. Faster Pit Stops Are Associated With Better Race Results

The scatter plot reveals a negative relationship between average pit stop duration and finishing position.

Insight:

Drivers with quicker pit stops generally tend to achieve better finishing positions, suggesting that pit stop performance contributes to race success.

Important Note:

The relationship is not perfect. Car performance, driver skill, and race strategy also influence finishing position.